
# 02-Compaint_Analysis
## From Complaint Screening to Evidence-Led Root Cause Investigation

### Objective
Use NHTSA vehicle complaints to identify recurring automotive failure signals, validate the patterns discovered through SQL, and prepare a reliable analytical dataset for Power BI.

### Investigation path
1. SQL screening identified Ford Escape as a candidate for deeper engine-related complaint analysis.
2. Model-year and narrative screening highlighted a potential coolant-intrusion / engine-failure pattern, particularly in 2017 complaints.
3. Python will validate the SQL findings, audit data quality, and quantify complaint-level symptom patterns.
4. Pareto analysis will prioritize reported failure signals for further investigation.
5. Power BI will communicate the findings through an interactive decision-support dashboard.

### Investigation hypothesis
2017 Ford Escape complaints may contain a recurring pattern linking reported coolant intrusion or coolant loss with misfire, overheating, and engine damage.

### Important limitations
- NHTSA complaints are owner-reported records, not independently verified failure diagnoses.
- Complaint counts are not vehicle failure rates; vehicle population and exposure data are needed to estimate rates.
- Narrative keyword matches are screening signals, not confirmed diagnoses.
- The analysis will preserve complaint identifiers and source categories to support traceability.


In [10]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the CSV:
DATA_PATH = Path(r"D:\Projects\SQL\Automotive Field Failure Intelligence\dataset\processed\NHTSA_Complaints_2020_2024_Profiled.csv")
df = pd.read_csv(DATA_PATH, low_memory=False)

df.columns = df.columns.str.strip().str.upper()

# Investigation scope established in SQL
TARGET_MAKE = "FORD"
TARGET_MODEL = "ESCAPE"
TARGET_YEAR = 2017

required_cols = ["CMPLID", "MAKETXT", "MODELTXT", "YEARTXT", "COMPDESC", "CDESCR"]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

print(f"Loaded {len(df):,} source rows.")
print("Investigation: 2017 Ford Escape — engine/coolant complaint signals.")


Loaded 418,884 source rows.
Investigation: 2017 Ford Escape — engine/coolant complaint signals.


In [17]:

# Normalize fields used in filtering
df["MAKE_N"] = df["MAKETXT"].astype("string").str.strip().str.upper()
df["MODEL_N"] = df["MODELTXT"].astype("string").str.strip().str.upper()
df["YEAR_N"] = pd.to_numeric(df["YEARTXT"], errors="coerce").astype("Int64")


escape = df[df["MAKE_N"].eq(TARGET_MAKE)& df["MODEL_N"].eq(TARGET_MODEL)].copy()

# SQL used distinct CMPLID counts
year_summary = (escape.groupby("YEAR_N").agg(complaint_rows=("CMPLID", "size"),distinct_complaints=("CMPLID", "nunique")).reset_index().sort_values("YEAR_N"))

print("Ford Escape complaint counts by model year:")
display(year_summary)

# Target-year scope
escape_2017 = escape[escape["YEAR_N"].eq(TARGET_YEAR)].copy()

print(f"2017 Ford Escape source rows: {len(escape_2017):,}")
print("2017 Ford Escape distinct complaint IDs:",escape_2017["CMPLID"].nunique())


Ford Escape complaint counts by model year:


,YEAR_N,complaint_rows,distinct_complaints
0,2001,8,8
1,2002,22,22
2,2003,30,30
3,2004,23,23
4,2005,72,72
5,2006,56,56
6,2007,51,51
7,2008,290,290
8,2009,442,442
9,2010,300,300


2017 Ford Escape source rows: 1,933
2017 Ford Escape distinct complaint IDs: 1933


In [16]:

narrative = escape["CDESCR"].fillna("").astype(str).str.upper()

symptom_patterns = {
    "overheating": r"OVERHEAT|OVER HEAT",
    "coolant_mentions": r"COOLANT|ANTIFREEZE",
    "water_pump_mentions": r"WATER PUMP",
    "engine_failure_mentions": r"ENGINE FAIL|ENGINE STALL|ENGINE SHUT"
}

escape["NARRATIVE_UPPER"] = narrative

for label, pattern in symptom_patterns.items():
    escape[label] = escape["NARRATIVE_UPPER"].str.contains(pattern, regex=True, na=False)

symptom_summary = (
    escape.groupby("YEAR_N")
    .agg(
        total_complaints=("CMPLID", "nunique"),
        **{
            name: (name, "sum")
            for name in symptom_patterns
        }
    )
    .reset_index()
    .sort_values("YEAR_N")
)

display(symptom_summary[symptom_summary["YEAR_N"].between(2013, 2020)])


,YEAR_N,total_complaints,overheating,coolant_mentions,water_pump_mentions,engine_failure_mentions
12,2013,1446,229,462,36,54
13,2014,1222,173,379,4,31
14,2015,648,45,113,5,14
15,2016,616,46,92,1,25
16,2017,1933,231,1032,22,172
17,2018,1194,180,692,10,105
18,2019,356,36,131,3,20
19,2020,1388,5,16,0,4


In [18]:

target_patterns = (
    r"COOLANT|ANTIFREEZE|OVERHEAT|OVER HEAT|"
    r"WATER PUMP|ENGINE FAIL|ENGINE STALL|ENGINE SHUT"
)

rca_review = escape_2017[escape_2017["CDESCR"].fillna("").astype(str).str.contains(target_patterns, case=False, regex=True)].copy()

review_cols = [c for c in ["CMPLID", "MODEL_YEAR", "COMPDESC", "CDESCR"] if c in rca_review.columns]

print(f"Rows selected for narrative review: {len(rca_review):,}")
print( "Distinct complaint IDs:", rca_review["CMPLID"].nunique())

display(rca_review[review_cols].head(15))


Rows selected for narrative review: 1,122
Distinct complaint IDs: 1122


,CMPLID,MODEL_YEAR,COMPDESC,CDESCR
18104,1651525,2017.0,ENGINE,TL* THE CONTACT OWNS A 2017 FORD ESCAPE. THE C...
20561,1653982,2017.0,ENGINE,TL* THE CONTACT OWNS A 2017 FORD ESCAPE. THE C...
26473,1659894,2017.0,ENGINE,"THIS WAS IN THE CITY, MORNING AND KIND OF RAIN..."
26474,1659895,2017.0,POWER TRAIN,"THIS WAS IN THE CITY, MORNING AND KIND OF RAIN..."
28189,1661610,2017.0,ENGINE,COOLANT LEAKING INTO CYLINDER AT 44K MILES ON ...
30940,1664361,2017.0,ENGINE AND ENGINE COOLING,TL* THE CONTACT OWNS A 2017 FORD ESCAPE. THE C...
33003,1666424,2017.0,ENGINE,VEHICLE WAS BEING DRIVEN AT HIGHWAY SPEED WITH...
36131,1669552,2017.0,ENGINE,THIS ISSUE OCCURRED WHILE PARKED IN THE DRIVEW...
36355,1669776,2017.0,ENGINE,"HAVE OWNED THE CAR FOR A MONTH, 65,000 MILES O..."
39544,1672965,2017.0,ENGINE,TL* THE CONTACT OWNS A 2017 FORD ESCAPE. THE C...



## SQL-to-Python Validation: Engineering Interpretation

The Python results reproduce the SQL model-year counts for Ford Escape, including 1,933 distinct complaints for model year 2017.

Keyword screening identifies 1,032 complaint rows mentioning coolant/antifreeze in 2017, alongside 231 overheating mentions and 172 engine-failure-related mentions. These indicators can overlap because a complaint may describe multiple symptoms.

From a plant / CFT (Cross-Functional Team) perspective, this pattern warrants investigation of the reported failure chain:

**Coolant loss or intrusion → possible cylinder misfire / rough running → overheating or loss of power → reported engine damage.**

The next step is to validate complaint-level uniqueness and quantify recurring signals before prioritizing them through Pareto analysis.

**Scope limitation:** NHTSA narratives are owner-reported complaints. These counts indicate reported signals, not confirmed defects, production-line root causes, or vehicle failure rates.


In [19]:

# Audit repeated complaint IDs within the 2017 Ford Escape cohort
id_check = (escape_2017.groupby("CMPLID", dropna=False).agg(row_count=("CMPLID", "size"),category_count=("COMPDESC", "nunique")))

print("2017 Ford Escape source rows:", len(escape_2017))
print("Distinct complaint IDs:", escape_2017["CMPLID"].nunique())
print("IDs with multiple source rows:", (id_check["row_count"] > 1).sum())
print("IDs mapped to multiple component categories:",(id_check["category_count"] > 1).sum())

display(id_check.query("row_count > 1").head(10))


2017 Ford Escape source rows: 1933
Distinct complaint IDs: 1933
IDs with multiple source rows: 0
IDs mapped to multiple component categories: 0


,row_count,category_count
CMPLID,,



### Complaint-Level Symptom Screening
Flag distinct Ford Escape complaints using predefined narrative keywords and quantify each symptom as a share of all distinct Ford Escape complaints. Symptoms may overlap within a complaint.


In [20]:

# One record per complaint ID for the Ford Escape cohort.
escape_unique = escape.drop_duplicates(subset="CMPLID").copy()

escape_unique["NARRATIVE_UPPER"] = (escape_unique["CDESCR"].fillna("").astype(str).str.upper())

symptom_patterns = {
    "Overheating": r"OVERHEAT|OVER HEAT",
    "Coolant / antifreeze": r"COOLANT|ANTIFREEZE",
    "Water pump": r"WATER PUMP",
    "Engine failure / stall": r"ENGINE FAIL|ENGINE STALL|ENGINE SHUT"
}

for symptom, pattern in symptom_patterns.items():
    col = symptom.lower().replace(" / ", "_").replace(" ", "_").replace("-", "_")
    escape_unique[col] = escape_unique["NARRATIVE_UPPER"].str.contains(pattern, regex=True, na=False)

symptom_cols = [symptom.lower().replace(" / ", "_").replace(" ", "_").replace("-", "_")for symptom in symptom_patterns]

symptom_summary = pd.DataFrame({"symptom": list(symptom_patterns.keys()),"complaint_count": [int(escape_unique[col].sum()) for col in symptom_cols]})

symptom_summary = symptom_summary.sort_values("complaint_count", ascending=False).reset_index(drop=True)

symptom_summary["pct_of_escape_complaints"] = (symptom_summary["complaint_count"]/ escape_unique["CMPLID"].nunique() * 100).round(2)

display(symptom_summary)


,symptom,complaint_count,pct_of_escape_complaints
0,Coolant / antifreeze,2943,26.81
1,Overheating,967,8.81
2,Engine failure / stall,435,3.96
3,Water pump,81,0.74



### Key Observation

Coolant/antifreeze mentions are the most frequent of the four screened signals, appearing in 2,943 complaints (26.81% of distinct Ford Escape complaints). Overheating appears in 967 (8.81%), engine failure/stall in 435 (3.96%), and water pump mentions in 81 (0.74%).

These are keyword-based screening results across all Ford Escape model years, not confirmed failure counts. A complaint can mention multiple symptoms, and reporting volume may vary by model year. The next step is to examine the 2017 cohort specifically and prioritize recurring signals through Pareto analysis.


In [21]:

def compare_symptom_counts(data, cohort_name):
    rows = []

    for symptom, pattern in symptom_patterns.items():
        matched = data[
            data["CDESCR"].fillna("").astype(str)
            .str.contains(pattern, case=False, regex=True, na=False)
        ]

        rows.append({
            "cohort": cohort_name,
            "symptom": symptom,
            "matching_rows": len(matched),
            "distinct_complaints": matched["CMPLID"].nunique()
        })

    return pd.DataFrame(rows)


comparison = pd.concat([
    compare_symptom_counts(escape, "All Ford Escape years"),
    compare_symptom_counts(escape_2017, "Ford Escape 2017")
], ignore_index=True)

display(comparison)


,cohort,symptom,matching_rows,distinct_complaints
0,All Ford Escape years,Overheating,967,967
1,All Ford Escape years,Coolant / antifreeze,2943,2943
2,All Ford Escape years,Water pump,81,81
3,All Ford Escape years,Engine failure / stall,435,435
4,Ford Escape 2017,Overheating,231,231
5,Ford Escape 2017,Coolant / antifreeze,1032,1032
6,Ford Escape 2017,Water pump,22,22
7,Ford Escape 2017,Engine failure / stall,172,172


### Pareto Analysis

#### Case Selection — Ford Escape, Model Year 2017

SQL screening identified the Ford Escape as the highest-volume model in the initial engine/cooling complaint analysis, with 3,665 complaints across 2001–2024. Across all Ford Escape model years in the current dataset, 2017 contributed 1,933 of 10,977 complaints (**17.61%**), the highest annual complaint count. This concentration motivated a focused investigation of recurring engine and cooling-related complaint signals.

**Limitation:** These are raw complaint counts and their share of recorded complaints, not failure rates. Vehicle population, sales, vehicle age, and reporting volume may influence the observed pattern.

In [22]:

component_pareto = (escape_2017.groupby("COMPDESC", dropna=False).agg(complaint_count=("CMPLID", "nunique")).reset_index()
.sort_values("complaint_count", ascending=False).reset_index(drop=True))

component_pareto["pct_of_complaints"] = (component_pareto["complaint_count"]/ component_pareto["complaint_count"].sum()* 100).round(2)

component_pareto["cumulative_pct"] = (component_pareto["pct_of_complaints"].cumsum()).round(2)

display(component_pareto.head(15))


,COMPDESC,complaint_count,pct_of_complaints,cumulative_pct
0,ENGINE,894,46.25,46.25
1,ENGINE AND ENGINE COOLING,296,15.31,61.56
2,POWER TRAIN,248,12.83,74.39
3,UNKNOWN OR OTHER,85,4.40,78.79
4,ELECTRICAL SYSTEM,63,3.26,82.05
5,WHEELS,42,2.17,84.22
6,STEERING,35,1.81,86.03
7,FUEL/PROPULSION SYSTEM,34,1.76,87.79
8,STRUCTURE:BODY,30,1.55,89.34
9,WHEELS:LUGS/NUTS/BOLTS/STUDS,25,1.29,90.63


### Key Observation

The top three reported component categories—ENGINE, ENGINE AND ENGINE COOLING, and POWER TRAIN—account for 74.39% of the 2017 Ford Escape complaint cohort. This concentration supports prioritizing powertrain-related narratives for further investigation. However, the categories reflect how complaints were classified; they do not independently confirm a specific component failure or root cause.

### Focused Investigation — Engine and Cooling Complaints

Narrow the 2017 Ford Escape cohort to complaints classified under ENGINE or ENGINE AND ENGINE COOLING. Screen their narratives for recurring coolant, overheating, water-pump, and engine-failure signals to identify cases for manual review. These keyword matches guide investigation; they do not establish confirmed root causes.

In [23]:

# Keep the two relevant component categories from the 2017 cohort
engine_cooling_categories = ["ENGINE","ENGINE AND ENGINE COOLING"]

engine_cooling_2017 = escape_2017[escape_2017["COMPDESC"].fillna("").str.strip().str.upper().isin(engine_cooling_categories)].copy()

# Define narrative signals
narrative = (engine_cooling_2017["CDESCR"].fillna("").astype(str).str.upper())

signal_patterns = {
    "coolant_antifreeze": r"COOLANT|ANTIFREEZE",
    "overheating": r"OVERHEAT|OVER HEAT",
    "water_pump": r"WATER PUMP",
    "engine_failure_stall": r"ENGINE FAIL|ENGINE STALL|ENGINE SHUT"
}

for signal, pattern in signal_patterns.items():
    engine_cooling_2017[signal] = narrative.str.contains(
        pattern, regex=True, na=False
    )

signal_cols = list(signal_patterns.keys())

engine_cooling_2017["signal_count"] = (
    engine_cooling_2017[signal_cols].sum(axis=1)
)

print("2017 Ford Escape complaints:", len(escape_2017))
print("Engine/cooling-category complaints:", len(engine_cooling_2017))
print("Complaints matching at least one signal:",int(engine_cooling_2017["signal_count"].gt(0).sum()))

display(engine_cooling_2017[signal_cols].sum().sort_values(ascending=False).rename("complaint_count").to_frame())


2017 Ford Escape complaints: 1933
Engine/cooling-category complaints: 1190
Complaints matching at least one signal: 967


,complaint_count
coolant_antifreeze,905
overheating,196
engine_failure_stall,139
water_pump,15


##### Observation
- 1,190 of 1,933 complaints (61.56%) fall under the two selected categories: ENGINE and ENGINE AND ENGINE COOLING.
- 967 of those 1,190 complaints (81.26%) match at least one of the four screened symptom signals.
- Coolant/antifreeze mentions lead with 905 complaints, followed by overheating (196), engine failure/stall (139), and water pump (15).

These signals overlap.

### Narrative Review — Candidate Root-Cause Signals

Extract complaints matching at least one predefined symptom keyword for manual review. Examine reported symptoms, operating conditions, mileage, and repair outcomes where available to assess recurring patterns and formulate a root-cause hypothesis. Narratives are owner-reported accounts and are not independently verified.

In [25]:

# Keep complaints matching at least one screened signal
rca_candidates = engine_cooling_2017[engine_cooling_2017["signal_count"].gt(0)].copy()

# Select useful context fields when available
preferred_cols = [
    "CMPLID",
    "MODEL_YEAR",
    "COMPDESC",
    "ODOMETER",
    "CDESCR",
    "coolant_antifreeze",
    "overheating",
    "water_pump",
    "engine_failure_stall",
    "signal_count"
]

review_cols = [col for col in preferred_cols if col in rca_candidates.columns]

rca_review = rca_candidates[review_cols].copy()

print("RCA candidate complaints:", len(rca_review))
print("Available review fields:", review_cols)

display(rca_review.sort_values(["signal_count", "CMPLID"],ascending=[False, True]).head(15))


RCA candidate complaints: 967
Available review fields: ['CMPLID', 'MODEL_YEAR', 'COMPDESC', 'CDESCR', 'coolant_antifreeze', 'overheating', 'water_pump', 'engine_failure_stall', 'signal_count']


,CMPLID,MODEL_YEAR,COMPDESC,CDESCR,coolant_antifreeze,overheating,water_pump,engine_failure_stall,signal_count
60760,1694181,2017.0,ENGINE,"MY CAR BEGAN LOSING POWER , COOLANT LEVEL DROP...",True,True,True,False,3
137449,1770870,2017.0,ENGINE,The contact owns a 2017 Ford Escape. The conta...,True,True,False,True,3
175905,1809326,2017.0,ENGINE,We discovered that our 2017 Ford Escape was le...,True,True,True,False,3
189413,1822834,2017.0,ENGINE,Coolant leaking into the engine cylinders - it...,True,True,False,True,3
191712,1825133,2017.0,ENGINE,The contact owns a 2017 Ford Escape. The conta...,True,True,False,True,3
202179,1835600,2017.0,ENGINE,The vehicle started over heating for no reason...,True,True,False,True,3
280941,1914362,2017.0,ENGINE,Check Engine light came on. Notice some rough ...,True,True,False,True,3
303897,1937318,2017.0,ENGINE,"I purchased my Ford escape 2017 ""new"" from Gai...",True,True,False,True,3
304986,1938407,2017.0,ENGINE,Hello - I had my 2017 Ford Escape SE Ecoboost ...,True,True,False,True,3
309512,1942933,2017.0,ENGINE,Cooling system-related engine failure. Coolant...,True,True,False,True,3


### Power BI Data Preparation and Export

Prepare three analysis-ready tables: a Ford Escape model-year trend, a 2017 component Pareto, and a complaint-level symptom investigation dataset. These tables will power the dashboard's trend, prioritization, and narrative drill-through views. Counts represent reported complaints, not verified failures or population-adjusted failure rates.

In [31]:

# 1. Model-year trend: complaint counts and symptom counts

trend_source = escape.copy()

# Create the symptom flags on this dataframe
narrative = trend_source["CDESCR"].fillna("").astype(str).str.upper()

trend_source["overheating"] = narrative.str.contains(
    r"OVERHEAT|OVER HEAT", regex=True, na=False
)
trend_source["coolant_mentions"] = narrative.str.contains(
    r"COOLANT|ANTIFREEZE", regex=True, na=False
)
trend_source["water_pump_mentions"] = narrative.str.contains(
    r"WATER PUMP", regex=True, na=False
)
trend_source["engine_failure_mentions"] = narrative.str.contains(
    r"ENGINE FAIL|ENGINE STALL|ENGINE SHUT", regex=True, na=False
)

trend_source["YEAR_N"] = pd.to_numeric(
    trend_source["YEAR_N"], errors="coerce"
)

trend_export = (
    trend_source.dropna(subset=["YEAR_N"])
    .groupby("YEAR_N")
    .agg(
        complaint_count=("CMPLID", "nunique"),
        overheating=("overheating", "sum"),
        coolant_mentions=("coolant_mentions", "sum"),
        water_pump_mentions=("water_pump_mentions", "sum"),
        engine_failure_mentions=("engine_failure_mentions", "sum")
    )
    .reset_index()
    .sort_values("YEAR_N")
)

trend_export["YEAR_N"] = trend_export["YEAR_N"].astype(int)

print("Model-year rows:", len(trend_export))
print("Total complaints:", trend_export["complaint_count"].sum())

display(trend_export)


Model-year rows: 24
Total complaints: 10978


,YEAR_N,complaint_count,overheating,coolant_mentions,water_pump_mentions,engine_failure_mentions
0,2001,8,0,0,0,0
1,2002,22,0,0,0,0
2,2003,30,0,0,0,0
3,2004,23,0,0,0,0
4,2005,72,0,0,0,0
5,2006,56,0,0,0,1
6,2007,51,2,3,0,3
7,2008,290,0,0,0,1
8,2009,442,4,0,0,0
9,2010,300,0,0,0,1


In [33]:

from pathlib import Path

OUTPUT_DIR = Path(r"D:\Projects\SQL\Automotive Field Failure Intelligence\powerbi\powerbi_exports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Export folder:", OUTPUT_DIR.resolve())



# 1. Model-year trend: complaint counts and symptom counts

trend_source = escape.copy()

# Create the symptom flags on this dataframe
narrative = trend_source["CDESCR"].fillna("").astype(str).str.upper()

trend_source["overheating"] = narrative.str.contains(
    r"OVERHEAT|OVER HEAT", regex=True, na=False
)
trend_source["coolant_mentions"] = narrative.str.contains(
    r"COOLANT|ANTIFREEZE", regex=True, na=False
)
trend_source["water_pump_mentions"] = narrative.str.contains(
    r"WATER PUMP", regex=True, na=False
)
trend_source["engine_failure_mentions"] = narrative.str.contains(
    r"ENGINE FAIL|ENGINE STALL|ENGINE SHUT", regex=True, na=False
)

trend_source["YEAR_N"] = pd.to_numeric(
    trend_source["YEAR_N"], errors="coerce"
)

trend_export = (
    trend_source.dropna(subset=["YEAR_N"])
    .groupby("YEAR_N")
    .agg(
        complaint_count=("CMPLID", "nunique"),
        overheating=("overheating", "sum"),
        coolant_mentions=("coolant_mentions", "sum"),
        water_pump_mentions=("water_pump_mentions", "sum"),
        engine_failure_mentions=("engine_failure_mentions", "sum")
    )
    .reset_index()
    .sort_values("YEAR_N")
)

trend_export["YEAR_N"] = trend_export["YEAR_N"].astype(int)

print("Model-year rows:", len(trend_export))
print("Total complaints:", trend_export["complaint_count"].sum())

display(trend_export)


# 2. Component Pareto for 2017
pareto_export = component_pareto.copy()

# 3. Complaint-level investigation table
investigation_cols = [
    col for col in [
        "CMPLID", "MODEL_YEAR", "COMPDESC", "CDESCR",
        "coolant_antifreeze", "overheating", "water_pump",
        "engine_failure_stall", "signal_count"
    ]
    if col in engine_cooling_2017.columns
]

investigation_export = (engine_cooling_2017[investigation_cols].copy())

# Add a convenient model-year field if needed
if "MODEL_YEAR" in investigation_export.columns:
    investigation_export["MODEL_YEAR"] = pd.to_numeric(
        investigation_export["MODEL_YEAR"], errors="coerce").astype("Int64")

# Exporting CSV files
trend_export.to_csv(OUTPUT_DIR / "ford_escape_model_year_trend.csv", index=False)
pareto_export.to_csv(       OUTPUT_DIR / "ford_escape_2017_component_pareto.csv", index=False)
investigation_export.to_csv(OUTPUT_DIR / "ford_escape_2017_investigation.csv", index=False)

print("Export complete:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"{file.name}: {len(pd.read_csv(file)):,} rows")


Export folder: D:\Projects\SQL\Automotive Field Failure Intelligence\powerbi\powerbi_exports
Model-year rows: 24
Total complaints: 10978


,YEAR_N,complaint_count,overheating,coolant_mentions,water_pump_mentions,engine_failure_mentions
0,2001,8,0,0,0,0
1,2002,22,0,0,0,0
2,2003,30,0,0,0,0
3,2004,23,0,0,0,0
4,2005,72,0,0,0,0
5,2006,56,0,0,0,1
6,2007,51,2,3,0,3
7,2008,290,0,0,0,1
8,2009,442,4,0,0,0
9,2010,300,0,0,0,1


Export complete:
ford_escape_2017_component_pareto.csv: 72 rows
ford_escape_2017_investigation.csv: 1,190 rows
ford_escape_model_year_trend.csv: 24 rows
